In [1]:
# %pip install -U git+https://github.com/graphcore-research/gfloat airium

<!-- # Copyright (c) 2024 Graphcore Ltd. All rights reserved. -->

# Making value tables

In this notebook, we generate value tables akin to those at [P3109](https://htmlpreview.github.io/?https://raw.githubusercontent.com/P3109/Public/main/Value%20Tables/html/index.html).

Thes tables comprise one-line summaries of each float value in the form
```text
Code Binary     = Exact binary E =  Float16 equivalent Float16 binary E    = Float Value
0x21 0_0100_001 = +0b1.001*2^-4  = 0_01011_0010000000 +0b1.0010000000*2^-4 = ~0.0703
```

In [2]:
import math
from gfloat import *
from gfloat.formats import *
import numpy as np
from IPython.display import HTML
import airium

## Define some helpers.

### Render with underscores separating s_e_m

E.g `0_1011_110`.  For formats with zero significand bits or zero exponent bits, we use `0_1011110_` or `0__10111110`.

In [3]:
def str_bits_with_underscores(fi, fv):
    signstr = f"{fv.signbit}_" if fi.is_signed else ""

    # 0_1011110_
    if fi.tSignificandBits == 0:
        return f"{signstr}{fv.exp:0{fi.expBits}b}_"

    # 0__1011110
    if fi.expBits == 0:
        return f"{signstr}{fv.significand:0{fi.tSignificandBits}b}"

    # 0_101_1110
    return f"{signstr}{fv.exp:0{fi.expBits}b}_{fv.significand:0{fi.tSignificandBits}b}"


fi = format_info_p3109(5, 3)
assert str_bits_with_underscores(fi, decode_float(fi, 0x09)) == "0_10_01"

fi = format_info_p3109(8, 1)
assert str_bits_with_underscores(fi, decode_float(fi, 0x41)) == "0_1000001_"

fi = format_info_p3109(8, 7)
assert str_bits_with_underscores(fi, decode_float(fi, 0x41)) == "0_1_000001"

### Render a binary16 value

Returns two strings, like this:
```
'0_00010_1010000000', '+0b1.1010000000*2^-13'
```

In [4]:
import struct


def b16_str(val) -> tuple[str, str]:
    """
    Represent VAL in binary16.

    If val does not convert exactly to binary16,
    returns "<Not16:{val}>"
    """
    with np.errstate(over="ignore"):
        b16 = np.float16(val)

    if float(b16) != val and np.isfinite(b16):
        # Finite, but not representable in float16
        return f"<Not16:{val}>", ""
    b16_int = struct.unpack("!H", struct.pack("!e", b16))[0]

    # bitstr is of the form 0_00000_1100000000
    s = f"{b16_int:016b}"
    e_str = s[1:6]
    m_str = s[6:]
    bitstr = f"{s[0]}_{e_str}_{m_str}"

    # pow2str is of the form '+0b0.1100000000*2^-15', or '' for nonfinite values
    e = int(e_str, 2) - 15
    m = int(m_str, 2)
    leading_bit = 0 if e == -15 else 1
    signstr = "-" if s[0] == "1" else "+"
    if np.isfinite(b16):
        pow2str = f"{signstr}0b{leading_bit}.{m:010b}*2^{e}"
    else:
        pow2str = ""
    return bitstr, pow2str


assert b16_str(13 * 2**-16) == ("0_00010_1010000000", "+0b1.1010000000*2^-13")

### Print one table row

In [5]:
def str_tablerow(fi, fv: FloatValue, show_b16_info=True, vs_width=14, vs_d=8):
    """
    Create a string of the form
      0x41 0_10000_01 = +0b1.01*2^0   = 1.25
    optionally adding binary16 info
      0x41 0_10000_01 = +0b1.01*2^0   = 0_01111_0100000000 +0b1.0100000000*2^0 = 1.25
    """
    text = []

    # 0x45 0_1000_101
    text.append(f"0x{fv.code:02x} {str_bits_with_underscores(fi, fv)}")

    finite_nonzero = np.isfinite(fv.fval) and fv.fval != 0

    #  = +0b1.101*2^-7 =
    if finite_nonzero:

        def signstr(fv):
            return "-" if fv.signbit else "+"

        b = "0" if fv.fclass == FloatClass.SUBNORMAL else "1"
        binary_pow2 = f"{signstr(fv)}0b{b}.{fv.significand:0{fi.tSignificandBits}b}*2^{fv.expval:<3}"
        text.append(binary_pow2)

    if show_b16_info and finite_nonzero:
        b16_binary_str, b16_bscistr = b16_str(fv.fval)
        text.append(f"{b16_binary_str} {b16_bscistr}")

    # 1.125
    text.append(float_tilde_unless_roundtrip_str(fv.fval, width=vs_width, d=vs_d))

    # Return tuple
    return " = ".join(text)


for fi in (format_info_p3109(8, 3), format_info_p3109(8, 1)):
    print(fi.name)
    for i in (
        0x00,
        0x01,
        0x07,
        0x21,
        0x40,
        0x41,
        0x7E,
        0x7F,
        0x80,
        0x81,
        0xE6,
        0xFE,
        0xFF,
    ):
        print(
            str_tablerow(fi, decode_float(fi, i), show_b16_info=True, vs_width=8, vs_d=4)
        )

p3109_8p3
0x00 0_00000_00 = 0.0
0x01 0_00000_01 = +0b0.01*2^-14 = 0_00000_0100000000 +0b0.0100000000*2^-15 = ~1.526e-05
0x07 0_00001_11 = +0b1.11*2^-14 = 0_00001_1100000000 +0b1.1100000000*2^-14 = ~0.0001
0x21 0_01000_01 = +0b1.01*2^-7  = 0_01000_0100000000 +0b1.0100000000*2^-7 = ~0.0098
0x40 0_10000_00 = +0b1.00*2^1   = 0_10000_0000000000 +0b1.0000000000*2^1 = 2.0
0x41 0_10000_01 = +0b1.01*2^1   = 0_10000_0100000000 +0b1.0100000000*2^1 = 2.5
0x7e 0_11111_10 = +0b1.10*2^16  = 0_11111_0000000000  = 98304.0
0x7f 0_11111_11 = inf
0x80 1_00000_00 = nan
0x81 1_00000_01 = -0b0.01*2^-14 = 1_00000_0100000000 -0b0.0100000000*2^-15 = ~-1.526e-05
0xe6 1_11001_10 = -0b1.10*2^10  = 1_11001_1000000000 -0b1.1000000000*2^10 = -1536.0
0xfe 1_11111_10 = -0b1.10*2^16  = 1_11111_0000000000  = -98304.0
0xff 1_11111_11 = -inf
p3109_8p1
0x00 0_0000000_ = 0.0
0x01 0_0000001_ = +0b1.0*2^-62 = <Not16:2.168404344971009e-19>  = ~2.168e-19
0x07 0_0000111_ = +0b1.0*2^-56 = <Not16:1.3877787807814457e-17>  = ~1.388e-

## Make HTML table

In [6]:
def mktbl(fi: FormatInfo, cols=4, skip_rows=None, airium_in=None, **kw):
    # Make tables
    nvals = 2**fi.bits
    rows = nvals // cols

    dark = True  # Used @media selector in CSS, but it doesn't work inside vscode

    def value_style(fv):
        if fv.fclass == FloatClass.SUBNORMAL:
            return "color: #0df" if dark else "color: #02b"

        if not (
            fv.fclass == FloatClass.NORMAL
            or fv.fclass == FloatClass.ZERO
            and not fv.signbit
        ):
            return "color: #d80" if dark else "color: #952"

        return ""

    title = f"FP8 Value Table, {fi.name}"
    if airium_in:
        a = airium_in
    else:
        a = airium.Airium()
        a.h2(_t=title)

    with a.table(
        klass="zmktbl",
        style="""margin: 0pt;
                                          border-collapse: collapse;""",
    ):
        for i in range(0, rows):
            tdstyle = "border: solid 2px #ccc;"
            if skip_rows and i in skip_rows:
                if i - 1 not in skip_rows:
                    with a.tr(klass="zmktbl"):
                        for x in range(cols):
                            a.td(_t="...", style=tdstyle + "text-align: center;")
                continue
            if i > 0 and i % 16 == 0:
                # blank row
                style = "height: 4ex; vertical-align: top;"
            else:
                style = "height: 0ex; vertical-align: top;"
            with a.tr(klass="zmktbl", style=style):
                for n in range(i, nvals, rows):
                    fv = decode_float(fi, n)
                    text = str_tablerow(fi, fv, show_b16_info=False, **kw)
                    with a.td(style=tdstyle + "text-align: left;"):
                        a.pre(
                            _t=text,
                            style=f"""
                              margin: 1pt 1pt 1pt 13pt; display: inline;
                              font-family: monospace;
                              font-size: 11;
                              font-weight: bold;
                              {value_style(fv)};
                              """,
                        )

    if airium_in:
        return a, title
    else:
        return str(a)


HTML(
    str(
        mktbl(
            format_info_ocp_e3m2,
            cols=2,
            skip_rows=set(range(5, 9)) | set(range(0x13, 0x1B)),
        )
    )
)

0x00 0_000_00 = 0.0,0x20 1_000_00 = -0.0
0x01 0_000_01 = +0b0.01*2^-2 = 0.0625,0x21 1_000_01 = -0b0.01*2^-2 = -0.0625
0x02 0_000_10 = +0b0.10*2^-2 = 0.125,0x22 1_000_10 = -0b0.10*2^-2 = -0.125
0x03 0_000_11 = +0b0.11*2^-2 = 0.1875,0x23 1_000_11 = -0b0.11*2^-2 = -0.1875
0x04 0_001_00 = +0b1.00*2^-2 = 0.25,0x24 1_001_00 = -0b1.00*2^-2 = -0.25
...,...
0x09 0_010_01 = +0b1.01*2^-1 = 0.625,0x29 1_010_01 = -0b1.01*2^-1 = -0.625
0x0a 0_010_10 = +0b1.10*2^-1 = 0.75,0x2a 1_010_10 = -0b1.10*2^-1 = -0.75
0x0b 0_010_11 = +0b1.11*2^-1 = 0.875,0x2b 1_010_11 = -0b1.11*2^-1 = -0.875
0x0c 0_011_00 = +0b1.00*2^0 = 1.0,0x2c 1_011_00 = -0b1.00*2^0 = -1.0
0x0d 0_011_01 = +0b1.01*2^0 = 1.25,0x2d 1_011_01 = -0b1.01*2^0 = -1.25


### OCP E2M1

This is a 4-bit format, without Inf or NaN

In [7]:
HTML(mktbl(format_info_ocp_e2m1, cols=1, vs_width=8, vs_d=3))

0x00 0_00_0 = 0.0
0x01 0_00_1 = +0b0.1*2^0 = 0.5
0x02 0_01_0 = +0b1.0*2^0 = 1.0
0x03 0_01_1 = +0b1.1*2^0 = 1.5
0x04 0_10_0 = +0b1.0*2^1 = 2.0
0x05 0_10_1 = +0b1.1*2^1 = 3.0
0x06 0_11_0 = +0b1.0*2^2 = 4.0
0x07 0_11_1 = +0b1.1*2^2 = 6.0
0x08 1_00_0 = -0.0
0x09 1_00_1 = -0b0.1*2^0 = -0.5
0x0a 1_01_0 = -0b1.0*2^0 = -1.0


## A range of 4-bit formats

In [8]:
def fourtables(k, p, **kw):
    a = airium.Airium()

    with a.table():
        for row in (0, 1):
            with a.tr():
                for signedness in ("u", "s"):
                    for domain in ("f", "e"):
                        has_infs = domain == "e"
                        is_signed = signedness == "s"
                        w = k - p + 1 if is_signed else k - p
                        bias = math.floor(2 ** (k - p - 1)) - 1
                        num_high_nans = 0 if is_signed else 1

                        name = f"binaryk{k}p{p}{signedness}{domain}"
                        fi = FormatInfo(
                            name,
                            k,
                            p,
                            bias=bias,
                            has_nz=False,
                            has_infs=has_infs,
                            num_high_nans=num_high_nans,
                            has_subnormals=True,
                            is_signed=is_signed,
                            is_twos_complement=False,
                        )
                        if row == 0:
                            a.th(_t=f"{name} bias={fi.bias}", style="text-align: center;")
                        else:
                            with a.td():
                                mktbl(fi, cols=1, airium_in=a, vs_width=8, vs_d=4, **kw)

    return HTML(str(a))


for p in (1, 2, 3, 4):
    display(fourtables(4, p))

### Now check unsigned vs signed, k=6

In [9]:
for p in (1, 2, 3, 6):
    display(fourtables(6, p, skip_rows=set(range(0x13, 0x1D)) | set(range(0x29, 0x3D))))

### IEEE P3109 4-bit formats

The IEEE P3109 interim report describes a family of formats parameterized by K and P, in which three 4-bit formats are defined.

The p=2 format is similar to OCP E2M1, with inf and nan:

In [10]:
HTML(mktbl(format_info_p3109(4, 2), cols=2, vs_width=8, vs_d=3))

0x00 0_00_0 = 0.0,0x08 1_00_0 = nan
0x01 0_00_1 = +0b0.1*2^0 = 0.5,0x09 1_00_1 = -0b0.1*2^0 = -0.5
0x02 0_01_0 = +0b1.0*2^0 = 1.0,0x0a 1_01_0 = -0b1.0*2^0 = -1.0
0x03 0_01_1 = +0b1.1*2^0 = 1.5,0x0b 1_01_1 = -0b1.1*2^0 = -1.5
0x04 0_10_0 = +0b1.0*2^1 = 2.0,0x0c 1_10_0 = -0b1.0*2^1 = -2.0
0x05 0_10_1 = +0b1.1*2^1 = 3.0,0x0d 1_10_1 = -0b1.1*2^1 = -3.0
0x06 0_11_0 = +0b1.0*2^2 = 4.0,0x0e 1_11_0 = -0b1.0*2^2 = -4.0
0x07 0_11_1 = inf,0x0f 1_11_1 = -inf


While the p=1 format is a "pure exponential" format with values 2^-2 to 2^3:

In [11]:
HTML(mktbl(format_info_p3109(4, 1), cols=2, vs_width=8, vs_d=3))

0x00 0_000_ = 0.0,0x08 1_000_ = nan
0x01 0_001_ = +0b1.0*2^-2 = 0.25,0x09 1_001_ = -0b1.0*2^-2 = -0.25
0x02 0_010_ = +0b1.0*2^-1 = 0.5,0x0a 1_010_ = -0b1.0*2^-1 = -0.5
0x03 0_011_ = +0b1.0*2^0 = 1.0,0x0b 1_011_ = -0b1.0*2^0 = -1.0
0x04 0_100_ = +0b1.0*2^1 = 2.0,0x0c 1_100_ = -0b1.0*2^1 = -2.0
0x05 0_101_ = +0b1.0*2^2 = 4.0,0x0d 1_101_ = -0b1.0*2^2 = -4.0
0x06 0_110_ = +0b1.0*2^3 = 8.0,0x0e 1_110_ = -0b1.0*2^3 = -8.0
0x07 0_111_ = inf,0x0f 1_111_ = -inf


And p=3, a linear format with values 0.25 * range(7)

In [12]:
HTML(mktbl(format_info_p3109(4, 3), cols=2, vs_width=8, vs_d=3))

0x00 0_0_00 = 0.0,0x08 1_0_00 = nan
0x01 0_0_01 = +0b0.01*2^1 = 0.5,0x09 1_0_01 = -0b0.01*2^1 = -0.5
0x02 0_0_10 = +0b0.10*2^1 = 1.0,0x0a 1_0_10 = -0b0.10*2^1 = -1.0
0x03 0_0_11 = +0b0.11*2^1 = 1.5,0x0b 1_0_11 = -0b0.11*2^1 = -1.5
0x04 0_1_00 = +0b1.00*2^1 = 2.0,0x0c 1_1_00 = -0b1.00*2^1 = -2.0
0x05 0_1_01 = +0b1.01*2^1 = 2.5,0x0d 1_1_01 = -0b1.01*2^1 = -2.5
0x06 0_1_10 = +0b1.10*2^1 = 3.0,0x0e 1_1_10 = -0b1.10*2^1 = -3.0
0x07 0_1_11 = inf,0x0f 1_1_11 = -inf


### OCP E2M3

This 6-bit format has 32 values, with no `NaN` or `Inf`, but does have `-0`.
The positive subnormals are the linear ramp of eighths: [n/8 for n in 1:7].

One might describe the format in text as:

> zero to one by eighths, two to four by quarters, four to eight by halves

where "to" is open-ended, or "to" is not "thru".

In [13]:
HTML(mktbl(format_info_ocp_e2m3, cols=2, vs_width=8, vs_d=3))

0x00 0_00_000 = 0.0,0x20 1_00_000 = -0.0
0x01 0_00_001 = +0b0.001*2^0 = 0.125,0x21 1_00_001 = -0b0.001*2^0 = -0.125
0x02 0_00_010 = +0b0.010*2^0 = 0.25,0x22 1_00_010 = -0b0.010*2^0 = -0.25
0x03 0_00_011 = +0b0.011*2^0 = 0.375,0x23 1_00_011 = -0b0.011*2^0 = -0.375
0x04 0_00_100 = +0b0.100*2^0 = 0.5,0x24 1_00_100 = -0b0.100*2^0 = -0.5
0x05 0_00_101 = +0b0.101*2^0 = 0.625,0x25 1_00_101 = -0b0.101*2^0 = -0.625
0x06 0_00_110 = +0b0.110*2^0 = 0.75,0x26 1_00_110 = -0b0.110*2^0 = -0.75
0x07 0_00_111 = +0b0.111*2^0 = 0.875,0x27 1_00_111 = -0b0.111*2^0 = -0.875
0x08 0_01_000 = +0b1.000*2^0 = 1.0,0x28 1_01_000 = -0b1.000*2^0 = -1.0
0x09 0_01_001 = +0b1.001*2^0 = 1.125,0x29 1_01_001 = -0b1.001*2^0 = -1.125
0x0a 0_01_010 = +0b1.010*2^0 = 1.25,0x2a 1_01_010 = -0b1.010*2^0 = -1.25


### OCP Formats: E5M2, E4M3

In [14]:
HTML(mktbl(format_info_ocp_e5m2, cols=4, skip_rows=(0x10, 0x30), vs_width=8, vs_d=5))

0x00 0_00000_00 = 0.0,0x40 0_10000_00 = +0b1.00*2^1 = 2.0,0x80 1_00000_00 = -0.0,0xc0 1_10000_00 = -0b1.00*2^1 = -2.0
0x01 0_00000_01 = +0b0.01*2^-14 = ~1.5259e-05,0x41 0_10000_01 = +0b1.01*2^1 = 2.5,0x81 1_00000_01 = -0b0.01*2^-14 = ~-1.5259e-05,0xc1 1_10000_01 = -0b1.01*2^1 = -2.5
0x02 0_00000_10 = +0b0.10*2^-14 = ~3.0518e-05,0x42 0_10000_10 = +0b1.10*2^1 = 3.0,0x82 1_00000_10 = -0b0.10*2^-14 = ~-3.0518e-05,0xc2 1_10000_10 = -0b1.10*2^1 = -3.0
0x03 0_00000_11 = +0b0.11*2^-14 = ~4.5776e-05,0x43 0_10000_11 = +0b1.11*2^1 = 3.5,0x83 1_00000_11 = -0b0.11*2^-14 = ~-4.5776e-05,0xc3 1_10000_11 = -0b1.11*2^1 = -3.5
0x04 0_00001_00 = +0b1.00*2^-14 = ~6.1035e-05,0x44 0_10001_00 = +0b1.00*2^2 = 4.0,0x84 1_00001_00 = -0b1.00*2^-14 = ~-6.1035e-05,0xc4 1_10001_00 = -0b1.00*2^2 = -4.0
0x05 0_00001_01 = +0b1.01*2^-14 = ~7.6294e-05,0x45 0_10001_01 = +0b1.01*2^2 = 5.0,0x85 1_00001_01 = -0b1.01*2^-14 = ~-7.6294e-05,0xc5 1_10001_01 = -0b1.01*2^2 = -5.0
0x06 0_00001_10 = +0b1.10*2^-14 = ~9.1553e-05,0x46 0_10001_10 = +0b1.10*2^2 = 6.0,0x86 1_00001_10 = -0b1.10*2^-14 = ~-9.1553e-05,0xc6 1_10001_10 = -0b1.10*2^2 = -6.0
0x07 0_00001_11 = +0b1.11*2^-14 = ~0.00011,0x47 0_10001_11 = +0b1.11*2^2 = 7.0,0x87 1_00001_11 = -0b1.11*2^-14 = ~-0.00011,0xc7 1_10001_11 = -0b1.11*2^2 = -7.0
0x08 0_00010_00 = +0b1.00*2^-13 = ~0.00012,0x48 0_10010_00 = +0b1.00*2^3 = 8.0,0x88 1_00010_00 = -0b1.00*2^-13 = ~-0.00012,0xc8 1_10010_00 = -0b1.00*2^3 = -8.0
0x09 0_00010_01 = +0b1.01*2^-13 = ~0.00015,0x49 0_10010_01 = +0b1.01*2^3 = 10.0,0x89 1_00010_01 = -0b1.01*2^-13 = ~-0.00015,0xc9 1_10010_01 = -0b1.01*2^3 = -10.0
0x0a 0_00010_10 = +0b1.10*2^-13 = ~0.00018,0x4a 0_10010_10 = +0b1.10*2^3 = 12.0,0x8a 1_00010_10 = -0b1.10*2^-13 = ~-0.00018,0xca 1_10010_10 = -0b1.10*2^3 = -12.0


In [15]:
HTML(mktbl(format_info_ocp_e4m3, cols=4, skip_rows=range(0x10, 0x30), vs_width=8, vs_d=5))

0x00 0_0000_000 = 0.0,0x40 0_1000_000 = +0b1.000*2^1 = 2.0,0x80 1_0000_000 = -0.0,0xc0 1_1000_000 = -0b1.000*2^1 = -2.0
0x01 0_0000_001 = +0b0.001*2^-6 = ~0.00195,0x41 0_1000_001 = +0b1.001*2^1 = 2.25,0x81 1_0000_001 = -0b0.001*2^-6 = ~-0.00195,0xc1 1_1000_001 = -0b1.001*2^1 = -2.25
0x02 0_0000_010 = +0b0.010*2^-6 = ~0.00391,0x42 0_1000_010 = +0b1.010*2^1 = 2.5,0x82 1_0000_010 = -0b0.010*2^-6 = ~-0.00391,0xc2 1_1000_010 = -0b1.010*2^1 = -2.5
0x03 0_0000_011 = +0b0.011*2^-6 = ~0.00586,0x43 0_1000_011 = +0b1.011*2^1 = 2.75,0x83 1_0000_011 = -0b0.011*2^-6 = ~-0.00586,0xc3 1_1000_011 = -0b1.011*2^1 = -2.75
0x04 0_0000_100 = +0b0.100*2^-6 = ~0.00781,0x44 0_1000_100 = +0b1.100*2^1 = 3.0,0x84 1_0000_100 = -0b0.100*2^-6 = ~-0.00781,0xc4 1_1000_100 = -0b1.100*2^1 = -3.0
0x05 0_0000_101 = +0b0.101*2^-6 = ~0.00977,0x45 0_1000_101 = +0b1.101*2^1 = 3.25,0x85 1_0000_101 = -0b0.101*2^-6 = ~-0.00977,0xc5 1_1000_101 = -0b1.101*2^1 = -3.25
0x06 0_0000_110 = +0b0.110*2^-6 = ~0.01172,0x46 0_1000_110 = +0b1.110*2^1 = 3.5,0x86 1_0000_110 = -0b0.110*2^-6 = ~-0.01172,0xc6 1_1000_110 = -0b1.110*2^1 = -3.5
0x07 0_0000_111 = +0b0.111*2^-6 = ~0.01367,0x47 0_1000_111 = +0b1.111*2^1 = 3.75,0x87 1_0000_111 = -0b0.111*2^-6 = ~-0.01367,0xc7 1_1000_111 = -0b1.111*2^1 = -3.75
0x08 0_0001_000 = +0b1.000*2^-6 = 0.015625,0x48 0_1001_000 = +0b1.000*2^2 = 4.0,0x88 1_0001_000 = -0b1.000*2^-6 = ~-0.01562,0xc8 1_1001_000 = -0b1.000*2^2 = -4.0
0x09 0_0001_001 = +0b1.001*2^-6 = ~0.01758,0x49 0_1001_001 = +0b1.001*2^2 = 4.5,0x89 1_0001_001 = -0b1.001*2^-6 = ~-0.01758,0xc9 1_1001_001 = -0b1.001*2^2 = -4.5
0x0a 0_0001_010 = +0b1.010*2^-6 = ~0.01953,0x4a 0_1001_010 = +0b1.010*2^2 = 5.0,0x8a 1_0001_010 = -0b1.010*2^-6 = ~-0.01953,0xca 1_1001_010 = -0b1.010*2^2 = -5.0


### IEEE WG P3109 KpP formats

We choose just one example: `p3109(k=8, p=3)`, which has the same number of exponent bits as OCP E5 

In [16]:
HTML(
    mktbl(
        format_info_p3109(8, 3), cols=4, skip_rows=range(0x10, 0x30), vs_width=8, vs_d=5
    )
)

0x00 0_00000_00 = 0.0,0x40 0_10000_00 = +0b1.00*2^1 = 2.0,0x80 1_00000_00 = nan,0xc0 1_10000_00 = -0b1.00*2^1 = -2.0
0x01 0_00000_01 = +0b0.01*2^-14 = ~1.5259e-05,0x41 0_10000_01 = +0b1.01*2^1 = 2.5,0x81 1_00000_01 = -0b0.01*2^-14 = ~-1.5259e-05,0xc1 1_10000_01 = -0b1.01*2^1 = -2.5
0x02 0_00000_10 = +0b0.10*2^-14 = ~3.0518e-05,0x42 0_10000_10 = +0b1.10*2^1 = 3.0,0x82 1_00000_10 = -0b0.10*2^-14 = ~-3.0518e-05,0xc2 1_10000_10 = -0b1.10*2^1 = -3.0
0x03 0_00000_11 = +0b0.11*2^-14 = ~4.5776e-05,0x43 0_10000_11 = +0b1.11*2^1 = 3.5,0x83 1_00000_11 = -0b0.11*2^-14 = ~-4.5776e-05,0xc3 1_10000_11 = -0b1.11*2^1 = -3.5
0x04 0_00001_00 = +0b1.00*2^-14 = ~6.1035e-05,0x44 0_10001_00 = +0b1.00*2^2 = 4.0,0x84 1_00001_00 = -0b1.00*2^-14 = ~-6.1035e-05,0xc4 1_10001_00 = -0b1.00*2^2 = -4.0
0x05 0_00001_01 = +0b1.01*2^-14 = ~7.6294e-05,0x45 0_10001_01 = +0b1.01*2^2 = 5.0,0x85 1_00001_01 = -0b1.01*2^-14 = ~-7.6294e-05,0xc5 1_10001_01 = -0b1.01*2^2 = -5.0
0x06 0_00001_10 = +0b1.10*2^-14 = ~9.1553e-05,0x46 0_10001_10 = +0b1.10*2^2 = 6.0,0x86 1_00001_10 = -0b1.10*2^-14 = ~-9.1553e-05,0xc6 1_10001_10 = -0b1.10*2^2 = -6.0
0x07 0_00001_11 = +0b1.11*2^-14 = ~0.00011,0x47 0_10001_11 = +0b1.11*2^2 = 7.0,0x87 1_00001_11 = -0b1.11*2^-14 = ~-0.00011,0xc7 1_10001_11 = -0b1.11*2^2 = -7.0
0x08 0_00010_00 = +0b1.00*2^-13 = ~0.00012,0x48 0_10010_00 = +0b1.00*2^3 = 8.0,0x88 1_00010_00 = -0b1.00*2^-13 = ~-0.00012,0xc8 1_10010_00 = -0b1.00*2^3 = -8.0
0x09 0_00010_01 = +0b1.01*2^-13 = ~0.00015,0x49 0_10010_01 = +0b1.01*2^3 = 10.0,0x89 1_00010_01 = -0b1.01*2^-13 = ~-0.00015,0xc9 1_10010_01 = -0b1.01*2^3 = -10.0
0x0a 0_00010_10 = +0b1.10*2^-13 = ~0.00018,0x4a 0_10010_10 = +0b1.10*2^3 = 12.0,0x8a 1_00010_10 = -0b1.10*2^-13 = ~-0.00018,0xca 1_10010_10 = -0b1.10*2^3 = -12.0


### Some tiny tiny formats

And finally, some tiny tiny formats.  We will take a P3109 format, and remove its infinities.

For p=1, we get as usual, a pure-exponential format with range 2^-1 to 2^1:

In [17]:
p3109_3p1f = format_info_p3109(3, 1)
p3109_3p1f.has_infs = False
HTML(mktbl(p3109_3p1f, cols=2, vs_width=8, vs_d=3))

0x00 0_00_ = 0.0,0x04 1_00_ = nan
0x01 0_01_ = +0b1.0*2^0 = 1.0,0x05 1_01_ = -0b1.0*2^0 = -1.0
0x02 0_10_ = +0b1.0*2^1 = 2.0,0x06 1_10_ = -0b1.0*2^1 = -2.0
0x03 0_11_ = +0b1.0*2^2 = 4.0,0x07 1_11_ = -0b1.0*2^2 = -4.0


And for p=2, we get, as usual for p=k-1, a purely linear format.  In this case, with values (0.5, 1.0, 1.5). Again as usual, 1.0 is in the middle of the range.

In [18]:
p3109_3p1f = format_info_p3109(3, 2)
p3109_3p1f.has_infs = False
HTML(mktbl(p3109_3p1f, cols=2, vs_width=8, vs_d=3))

0x00 0_0_0 = 0.0,0x04 1_0_0 = nan
0x01 0_0_1 = +0b0.1*2^1 = 1.0,0x05 1_0_1 = -0b0.1*2^1 = -1.0
0x02 0_1_0 = +0b1.0*2^1 = 2.0,0x06 1_1_0 = -0b1.0*2^1 = -2.0
0x03 0_1_1 = +0b1.1*2^1 = 3.0,0x07 1_1_1 = -0b1.1*2^1 = -3.0
